Death Valley: contribution to the development collection

Add the region selected before acquisition in the multi-region plan. Compare
unchanged lunar queries against the original three Earth sites and Death Valley.
Preserve 300/600/960 m windows, both lunar support views and both Earth eligibility
cohorts. This is development EDA, not independent model or analog validation.

In [ ]:
import hashlib
import json
import sys
from datetime import UTC, datetime
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import nbformat
import numpy as np
import pandas as pd
import rasterio
from IPython.display import Image, Markdown, display
from matplotlib.figure import Figure

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'outline.md').exists())
sys.path.insert(0, str(ROOT / 'notebooks'))
from inspection import check_input, file_hash, verify_run
from patch_analysis import measure_patch, measure_planarity
from retrieval_analysis import terrain_discrepancies

RUN_ID = '20261029T075850772998Z_b58cb9'
RUN = ROOT / 'data/processed/earth_terrain_expansion' / RUN_ID
PREVIOUS = ROOT / 'notebooks/diagnostics/scale-eda-20261029T012946475712Z'
NOTEBOOK = ROOT / 'notebooks/exploration/20_death_valley_contribution_eda.ipynb'
OUT = ROOT / 'notebooks/diagnostics' / ('death-valley-eda-' + datetime.now(UTC).strftime('%Y%m%dT%H%M%S%fZ'))
OUT.mkdir(parents=True, exist_ok=False)
expected = verify_run(RUN)
evidence = {str(RUN / 'provenance.json'): file_hash(RUN / 'provenance.json'),
            str(PREVIOUS / 'provenance.json'): file_hash(PREVIOUS / 'provenance.json')}
previous_manifest = json.loads((PREVIOUS / 'provenance.json').read_text())
assert previous_manifest['status'] == 'complete'
expected.update({str((PREVIOUS / name).resolve()): digest for name, digest in previous_manifest['outputs'].items()})
old = pd.read_csv(check_input(PREVIOUS / 'center_metrics.csv', expected, evidence))
quality = json.loads(check_input(RUN / 'earth/death_valley/quality.json', expected, evidence).read_text())
assert quality['joint_feature_fraction'] == quality['elevation_coverage'] == quality['valid_input_fraction'] == 1
assert quality['screened_cells_missing_features'] == 0
CHANNELS = ['elevation_m', 'relative_elevation_m', 'slope_deg', 'laplacian_per_m', 'local_elevation_std_m']
FEATURES = ['slope_median_deg', 'slope_p90_deg', 'central_relief_m', 'plane_residual_fraction', 'local_sd_median_m']
SIZES = [300, 600, 960]
SCENARIOS = ['baseline', 'with_planarity', 'with_local_sd']
layers = {}
grid = None
for channel in [*CHANNELS, 'natural_screen_mask', 'bare_sparse_fraction', 'built_fraction', 'crop_fraction', 'water_fraction']:
    path = RUN / 'earth/death_valley/grid_30m' / f'{channel}.tif'
    with rasterio.open(check_input(path, expected, evidence)) as src:
        current = (src.shape, src.transform, src.crs)
        assert grid is None or current == grid
        grid = current
        layers[channel] = src.read(1, masked=True).astype('float32').filled(np.nan)
assert grid is not None
shape, transform, crs = grid
assert transform.a == 30 and transform.e == -30 and crs is not None
terrain_eligible = np.logical_and.reduce([np.isfinite(layers[c]) for c in CHANNELS])
eligible = (layers['natural_screen_mask'] == 1) & terrain_eligible
source = '\n'.join(c.source for c in nbformat.read(NOTEBOOK, as_version=4).cells)
provenance: dict[str, Any] = {
    'status': 'running', 'notebook': str(NOTEBOOK.relative_to(ROOT)),
    'notebook_source_sha256': hashlib.sha256(source.encode()).hexdigest(), 'processing_run': RUN_ID,
    'previous_eda': str(PREVIOUS.relative_to(ROOT)), 'region': 'death_valley', 'role': 'development',
    'helpers': {name: file_hash(ROOT / 'notebooks' / name) for name in ['inspection.py', 'patch_analysis.py', 'retrieval_analysis.py']},
    'sizes_m': SIZES, 'center_origin_row_col': [16, 16], 'stride_cells': 32,
    'earth_cohorts': ['common_960', 'eligible_at_size'], 'lunar_queries': 'fixed notebook16 common-960 cohort in both comparisons',
    'scenarios': SCENARIOS, 'features': FEATURES, 'bands': 'unchanged notebook17 physical bands',
    'interpretation': 'collection coverage, not relevance labels or model performance',
}
(OUT / 'provenance.json').write_text(json.dumps(provenance, indent=2), encoding='utf-8')


def save_figure(fig: Figure, name: str) -> None:
    fig.savefig(OUT / f'{name}.png', bbox_inches='tight')
    fig.savefig(OUT / f'{name}.pdf', bbox_inches='tight')
    display(Image(filename=str(OUT / f'{name}.png')))
    plt.close(fig)


print('Output:', OUT)

# Same source, screen and center-lattice policy

Both terrain and land-cover inputs are complete within the requested ROI; the
natural screen still excludes nonqualifying surfaces. Local elevation SD uses
11×11 cells (330 m). A common center needs a complete eligible 960 m window.
Native land-cover tiles were mosaicked before area aggregation; class codes
were never averaged. The 36°N boundary is covered from both source rows.

In [ ]:
records = []
screen_records = []
for center_row in range(16, shape[0] - 15, 32):
    for center_col in range(16, shape[1] - 15, 32):
        common = bool(eligible[center_row - 16:center_row + 16, center_col - 16:center_col + 16].all())
        for size in SIZES:
            side = size // 30
            row, col = center_row - side // 2, center_col - side // 2
            window = np.s_[row:row + side, col:col + side]
            if not terrain_eligible[window].all():
                continue
            x, y = transform * (center_col, center_row)
            record = {'body': 'earth', 'region': 'death_valley', 'population': 'natural_screen',
                            'size_m': size, 'center_id': f'death_valley:natural_screen:{center_row}:{center_col}',
                            'center_row': center_row, 'center_col': center_col, 'center_x_m': x, 'center_y_m': y,
                            'row': row, 'col': col, 'common': common,
                            **measure_patch(layers['elevation_m'][window], layers['slope_deg'][window], layers['local_elevation_std_m'][window]),
                            **measure_planarity(layers['elevation_m'][window])}
            retained = bool(eligible[window].all())
            screen_records.append({**record, 'population': 'terrain_complete_diagnostic', 'natural_complete': retained,
                                   'natural_cell_fraction': float(eligible[window].mean()),
                                   'bare_below_80pct_cells': int((layers['bare_sparse_fraction'][window] < .8).sum()),
                                   'built_positive_cells': int((layers['built_fraction'][window] > 0).sum()),
                                   'crop_positive_cells': int((layers['crop_fraction'][window] > 0).sum()),
                                   'water_positive_cells': int((layers['water_fraction'][window] > 0).sum())})
            if retained:
                records.append(record)
new = pd.DataFrame(records)
assert not new.duplicated(['center_id', 'size_m']).any() and np.isfinite(new[FEATURES]).all().all()
new.to_csv(OUT / 'new_center_metrics.csv', index=False)
inventory = new.groupby('size_m').agg(eligible_centers=('center_id', 'size'), common_centers=('common', 'sum')).reset_index()
inventory.to_csv(OUT / 'inventory.csv', index=False)
all_earth = pd.concat([old.loc[old['body'] == 'earth'], new], ignore_index=True)
quantiles = all_earth.groupby(['region', 'size_m'])[FEATURES].median().reset_index()
quantiles.to_csv(OUT / 'eligible_site_medians.csv', index=False)
display(inventory)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, channel, label, limits in zip(axes, ['elevation_m', 'slope_deg', 'natural_screen_mask'],
                                      ['Elevation (m)', 'Slope (degrees)', 'Natural screen'],
                                      [(None, None), (0, 40), (0, 1)], strict=True):
    values = layers[channel]
    im = ax.imshow(values, extent=(0, shape[1] * .03, -shape[0] * .03, 0),
                   cmap='terrain' if channel == 'elevation_m' else 'viridis', vmin=limits[0], vmax=limits[1])
    ax.set(title=label, xlabel='East from grid origin (km)', ylabel='North from grid origin (km)')
    fig.colorbar(im, ax=ax, shrink=.7)
fig.suptitle('Death Valley: processed ROI; source spacing 30 m; screen is not field suitability')
fig.tight_layout()
save_figure(fig, 'death_valley_maps')

# Findings-led check: what does the natural screen exclude?

The first execution added little strict-view coverage. Maps suggested that many
steeper patches fail the screen. Compare all complete-terrain windows on the same
lattice with retained and excluded windows. This diagnostic does not admit excluded
patches into retrieval or change the 80% bare/sparse threshold. Class-failure counts
can overlap; they describe the source map, not independently verified land cover.

In [ ]:
screen_diagnostic = pd.DataFrame(screen_records)
screen_diagnostic.to_csv(OUT / 'screen_diagnostic_centers.csv', index=False)
screen_summary_rows = []
for size, group in screen_diagnostic.groupby('size_m'):
    for label, part in [('all_complete_terrain', group), ('retained', group.loc[group['natural_complete']]),
                        ('excluded', group.loc[~group['natural_complete']])]:
        screen_summary_rows.append({'size_m': size, 'population': label, 'centers': len(part),
                                    'slope_p90_median_deg': part['slope_p90_deg'].median(),
                                    'central_relief_median_m': part['central_relief_m'].median(),
                                    'plane_fraction_median': part['plane_residual_fraction'].median(),
                                    'p90_slope_at_least_15deg_centers': int((part['slope_p90_deg'] >= 15).sum()),
                                    'any_bare_failure_centers': int((part['bare_below_80pct_cells'] > 0).sum()),
                                    'any_built_failure_centers': int((part['built_positive_cells'] > 0).sum()),
                                    'any_crop_failure_centers': int((part['crop_positive_cells'] > 0).sum()),
                                    'any_water_failure_centers': int((part['water_positive_cells'] > 0).sum())})
screen_summary = pd.DataFrame(screen_summary_rows)
screen_summary.to_csv(OUT / 'screen_selection_summary.csv', index=False)
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
six_hundred = screen_diagnostic.loc[screen_diagnostic['size_m'] == 600]
for ax, feature, label in zip(axes, ['slope_p90_deg', 'plane_residual_fraction'],
                               ['P90 slope (degrees)', 'Plane residual fraction'], strict=True):
    for name, group in [('All complete terrain', six_hundred), ('Retained natural screen', six_hundred.loc[six_hundred['natural_complete']]),
                        ('Excluded by screen', six_hundred.loc[~six_hundred['natural_complete']])]:
        values = np.sort(group[feature])
        ax.plot(values, np.arange(1, len(values) + 1) / len(values), label=name)
    ax.set(xlabel=label, ylabel='Within-group fraction', ylim=(0, 1))
axes[0].legend(fontsize=8)
fig.suptitle('600 m: selection diagnostic only; excluded windows are not added to the candidate pool')
fig.tight_layout()
save_figure(fig, '600m_screen_selection')

# Fixed lunar queries, changing Earth collection

The lunar queries remain the same common-960 centers in every comparison.
Thus switching Earth eligibility cohorts changes candidates alone. At each size
compare the original three-site pool with that pool plus Death Valley. A fixed
score ≤1 means all exploratory bands hold. Adding candidates cannot decrease
coverage; gains describe availability, not model quality. Count geographic
replication separately from newly covered queries. No query or final reserve
is selected because of its outcome.

In [ ]:
query_records = []
queries = old.loc[(old['body'] == 'moon') & old['common']].copy()
for cohort in ['common_960', 'eligible_at_size']:
    earth = all_earth.loc[all_earth['common']].copy() if cohort == 'common_960' else all_earth.copy()
    for size in SIZES:
        candidate_groups = {region: group.reset_index(drop=True) for region, group in
                            earth.loc[earth['size_m'] == size].groupby('region')}
        arrays = {region: group[FEATURES].to_numpy(dtype=float) for region, group in candidate_groups.items()}
        for query in queries.loc[queries['size_m'] == size].to_dict('records'):
            values = np.array([query[f] for f in FEATURES], dtype=float)
            scores = {region: terrain_discrepancies(values, candidates) for region, candidates in arrays.items()}
            for scenario in SCENARIOS:
                old_regions = sorted(region for region in scores if region != 'death_valley')
                old_sites = sum(bool((scores[region][scenario] <= 1).any()) for region in old_regions)
                new_scores = scores['death_valley'][scenario]
                best_index = int(np.argmin(new_scores))
                best = candidate_groups['death_valley'].iloc[best_index]
                old_best = min(float(scores[region][scenario].min()) for region in old_regions)
                new_best = float(new_scores[best_index])
                query_records.append({'cohort': cohort, 'size_m': size, 'scenario': scenario,
                                      'query_id': f'{query["center_id"]}:{size}', 'region': query['region'],
                                      'population': query['population'], 'old_sites_in_band': old_sites,
                                      'new_site_in_band': int(new_best <= 1), 'expanded_sites_in_band': old_sites + int(new_best <= 1),
                                      'old_best_score': old_best, 'new_best_score': new_best,
                                      'expanded_best_score': min(old_best, new_best),
                                      'best_new_candidate_id': f'{best["center_id"]}:{size}',
                                      'new_candidates': len(new_scores), 'new_candidates_in_band': int((new_scores <= 1).sum())})
results = pd.DataFrame(query_records)
results.to_csv(OUT / 'query_contributions.csv.gz', index=False, compression='gzip')
summaries = []
for key, population_frame in results.groupby(['cohort', 'size_m', 'scenario', 'population']):
    assert isinstance(key, tuple)
    cohort, size, scenario, population = key
    for region, group in [('ALL_QUERY_WEIGHTED', population_frame), *list(population_frame.groupby('region'))]:
        summaries.append({'cohort': cohort, 'size_m': size, 'scenario': scenario, 'population': population,
                          'region': region, 'queries': len(group),
                          'old_coverage_pct': 100 * (group['old_sites_in_band'] > 0).mean(),
                          'expanded_coverage_pct': 100 * (group['expanded_sites_in_band'] > 0).mean(),
                          'newly_covered_pct': 100 * ((group['old_sites_in_band'] == 0) & (group['new_site_in_band'] == 1)).mean(),
                          'new_site_coverage_pct': 100 * group['new_site_in_band'].mean(),
                          'old_multisite_pct': 100 * (group['old_sites_in_band'] >= 2).mean(),
                          'expanded_multisite_pct': 100 * (group['expanded_sites_in_band'] >= 2).mean()})
summary = pd.DataFrame(summaries)
summary.to_csv(OUT / 'coverage.csv', index=False)
macro = summary.loc[summary['region'] != 'ALL_QUERY_WEIGHTED'].groupby(
    ['cohort', 'size_m', 'scenario', 'population']).mean(numeric_only=True).drop(columns='queries').reset_index()
macro.to_csv(OUT / 'coverage_region_macro.csv', index=False)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
for ax, population in zip(axes, ['full_context', 'image_supported'], strict=True):
    values = summary.loc[(summary['region'] == 'ALL_QUERY_WEIGHTED') & (summary['size_m'] == 600)
                         & (summary['scenario'] == 'baseline') & (summary['population'] == population)].set_index('cohort')
    x = np.arange(2)
    for offset, metric, label in [(-.25, 'old_coverage_pct', 'Original three'), (0, 'expanded_coverage_pct', 'With Death Valley'),
                                 (.25, 'expanded_multisite_pct', 'At least two sites')]:
        ax.bar(x + offset, values.loc[['common_960', 'eligible_at_size'], metric], .25, label=label)
    ax.set(xticks=x, xticklabels=['Common 960 m', 'Eligible at 600 m'], title=population.replace('_', ' '),
           ylabel='Fixed lunar queries (%)', ylim=(0, 100))
axes[1].legend(fontsize=8)
fig.suptitle('600 m baseline: candidate availability and geographic replication')
fig.tight_layout()
save_figure(fig, '600m_collection_contribution')

# Keep the physical distributions visible

Curves are within-site fractions of eligible 600 m centers; each Earth site has
its own curve despite different counts. The lunar curve pools fixed strict-view
query centers, so it is not a macro average. Low plane residual fraction alone
is neither low absolute roughness nor evidence of source accuracy.

In [ ]:
# Read the Markdown notes below for the standalone report export.
report_notebook = json.loads((ROOT / 'notebooks/exploration/20_death_valley_contribution_eda.ipynb').read_text(encoding="utf-8"))
report_notes = {entry["metadata"]["report_section"]: "".join(entry["source"])
                for entry in report_notebook["cells"]
                if entry["cell_type"] == "markdown" and "report_section" in entry["metadata"]}
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, feature, label in zip(axes, ['slope_p90_deg', 'central_relief_m', 'plane_residual_fraction'],
                               ['P90 slope (degrees)', 'Central relief (m)', 'Plane residual fraction'], strict=True):
    for region, group in all_earth.loc[all_earth['size_m'] == 600].groupby('region'):
        values = np.sort(group[feature].to_numpy())
        ax.plot(values, np.arange(1, len(values) + 1) / len(values), label=str(region).replace('_', ' '),
                linewidth=2.5 if region == 'death_valley' else 1)
    values = np.sort(queries.loc[(queries['size_m'] == 600) & (queries['population'] == 'image_supported'), feature])
    ax.plot(values, np.arange(1, len(values) + 1) / len(values), '--', color='black', label='Moon fixed strict queries')
    ax.set(xlabel=label, ylabel='Fraction of centers', ylim=(0, 1))
axes[0].legend(fontsize=7)
fig.tight_layout()
save_figure(fig, '600m_distributions')

report = [
    *report_notes['narrative-c8b8973e-1'].split('\n'),
    f'Processing run: `{RUN_ID}`. The ROI has complete elevation, five-channel terrain features and land-cover input support. Four FABDEM and four WorldCover tiles cover the processing footprint. Native class mosaicking preserves seam contributions before fraction aggregation.',
    '',
    f"The unchanged natural screen retains {quality['natural_screen_fraction']:.1%} of ROI cells; no screened cell is missing a terrain feature. This screen is not a field-access or analog-quality judgment.",
    '',
    '## Candidate inventory',
    '',
    '| Size (m) | Eligible centers | Common-960 centers |',
    '| ---: | ---: | ---: |',
]
for row in inventory.to_dict('records'):
    report.append(f'| {row["size_m"]} | {row["eligible_centers"]} | {row["common_centers"]} |')
report += [
    *report_notes['narrative-c8b8973e-2'].split('\n'),
]
for row in summary.loc[(summary['size_m'] == 600) & (summary['region'] == 'ALL_QUERY_WEIGHTED')].to_dict('records'):
    report.append(f'| {row["cohort"]} | {row["population"]} | {row["scenario"]} | '
                  f'{row["old_coverage_pct"]:.1f} | {row["expanded_coverage_pct"]:.1f} | {row["expanded_multisite_pct"]:.1f} |')
report += [
    *report_notes['narrative-c8b8973e-3'].split('\n'),
]
for row in screen_summary.loc[screen_summary['size_m'] == 600].to_dict('records'):
    report.append(f'| {row["population"]} | {row["centers"]} | {row["slope_p90_median_deg"]:.2f} | '
                  f'{row["central_relief_median_m"]:.2f} | {row["p90_slope_at_least_15deg_centers"]} |')
report += [
    '',
    'The [screen summary](screen_selection_summary.csv) records overlapping failure reasons. Bare/sparse class fractions come from WorldCover; failure does not by itself establish why a surface is unsuitable. The full-terrain diagnostic remains outside retrieval. Review the purpose and selection effects of this Earth screen before treating a low contribution as evidence that the entire region lacks relevant terrain. Do not tune the threshold solely to improve lunar band coverage.',
    *report_notes['narrative-c8b8973e-4'].split('\n'),
    '- [New center metrics](new_center_metrics.csv), [inventory](inventory.csv), [site medians](eligible_site_medians.csv).',
    '- [Screen diagnostic centers](screen_diagnostic_centers.csv), [selection summary](screen_selection_summary.csv), [selection curves](600m_screen_selection.png).',
    '- [Per-query contributions](query_contributions.csv.gz), [regional and pooled coverage](coverage.csv), [region macro averages](coverage_region_macro.csv).',
    '- [Terrain and screen](death_valley_maps.png), [collection contribution](600m_collection_contribution.png), [distributions](600m_distributions.png).',
    '- [Provenance](provenance.json). All three sizes and both lunar views remain available in the tables.',
]
(OUT / 'REPORT.md').write_text('\n'.join(report) + '\n', encoding='utf-8')
provenance['inputs'] = evidence
provenance['outputs'] = {str(p.relative_to(OUT)): file_hash(p) for p in sorted(OUT.rglob('*')) if p.is_file() and p.name != 'provenance.json'}
provenance['status'] = 'complete'
(OUT / 'provenance.json').write_text(json.dumps(provenance, indent=2), encoding='utf-8')
display(Markdown('\n'.join(report)))
print('Complete:', OUT)

# Death Valley development contribution

A new geographically separated development region, selected before acquisition. No model, normalization or analog relevance labels were fitted.

## Processing acceptance



Same center lattice as notebook16: origin16, stride32cells; all largest windows fit the array. These are spatially correlated centers, not independent samples.

## 600 m fixed-query comparison

| Earth cohort | Lunar view | Scenario | Original coverage (%) | Expanded coverage (%) | Expanded ≥2-site coverage (%) |
| --- | --- | --- | ---: | ---: | ---: |


Queries are identical between Earth-cohort scenarios. Eligibility changes therefore apply to Earth candidates alone. Candidate counts differ; geographic replication and new availability do not establish training generalization or independent analog relevance. The fixed bands were inherited from notebook17 without tuning for Death Valley. Adding candidates cannot lower coverage.

## Findings-led screen diagnosis

The first execution showed little additional strict-view coverage. Maps motivated a same-lattice comparison before and after the natural screen, without changing the candidate pools.

| 600 m population | Centers | Median P90 slope (deg) | Median relief (m) | P90 slope ≥15° centers |
| --- | ---: | ---: | ---: | ---: |


## Next decision

Inspect the new regional distribution and remaining no-counterpart cases before deciding the next development sampling change. Wadi Rum remains the next planned geographically distant development region. Keep validation and final reserves closed to exploratory patch selection. Relevance assessment and source-quality interpretation remain necessary before learned retrieval.

## Artifacts
